# Data Cleaning and EDA

- The objective of this project is to predict rating for restaurants when no reviews / users votes are avaliable.
- The main motivation is that Yelp can use the predction of rating for new restaurants to formulate its search/recommendation ranking.
- This can further ensure exposure of new restaurants with high expected service quality.

## Set Up

In [1]:
# import packages
import polars as pl
import pandas as pd
from pathlib import Path
from yelp_predict.data_wrangling import expand_nested, expand_cats, col_dtype

In [2]:
# set the path to the datasets
data_path = Path().resolve().parent/"data"
business_path = data_path/"yelp_academic_dataset_business.json"

## Data Loading

In [3]:
# load the dataset
raw = pd.read_json(business_path, lines=True)

# get an overview on dataset columns and dtypes
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   business_id   150346 non-null  object 
 1   name          150346 non-null  object 
 2   address       150346 non-null  object 
 3   city          150346 non-null  object 
 4   state         150346 non-null  object 
 5   postal_code   150346 non-null  object 
 6   latitude      150346 non-null  float64
 7   longitude     150346 non-null  float64
 8   stars         150346 non-null  float64
 9   review_count  150346 non-null  int64  
 10  is_open       150346 non-null  int64  
 11  attributes    136602 non-null  object 
 12  categories    150243 non-null  object 
 13  hours         127123 non-null  object 
dtypes: float64(3), int64(2), object(9)
memory usage: 16.1+ MB


## Data Wrangling

- As this project aims to predict rating of resturants, I filter out observations that are not resturants

In [4]:
# filter by the condition that the categories column contain "Restaurants" (this project only predict rating for restaurant)
raw = pd.read_json(business_path, lines=True)
intermediate = raw[raw["categories"].str.contains("Restaurants", na=False)]

- Count the number of restaurants by city. This project focus on rating of restaurants in "Philadelphia", "Tampa", "Indianapolis", "Nashville", and "Tucson".
- These are the  5 cities with highest restaurant number.

In [5]:
# get the count of restaurants by city
city_counts = (
    intermediate
    .groupby(by="city")
    .size()
    .reset_index(name="counts")
    .sort_values(by="counts", ascending = False)
    .head(10)
    )
print(city_counts)

# keep restaurants in the top 5 cities
cities = ["Philadelphia", "Tampa", "Indianapolis", "Nashville", "Tucson"]
intermediate = intermediate[intermediate["city"].isin(cities)].reset_index()

             city  counts
577  Philadelphia    5852
761         Tampa    2960
344  Indianapolis    2862
508     Nashville    2502
801        Tucson    2466
515   New Orleans    2259
206      Edmonton    2166
655   Saint Louis    1790
619          Reno    1286
63          Boise     847


- Business attributes and openning hours are stored in json objects within column values
- Hence, there is a need to extract these values from the columns with nested json object
- For each business attribute extracted, a new column is created; same applies to openning hour from Mon to Sun

In [6]:
# each value in the attribute and hours columns is a json object that contains business relevant features
print(f"Example of value in attributes column: {intermediate["attributes"][10]}")
print(f"Example of value in hours column: {intermediate["hours"][10]}")

# hence, we expand the nested json objects in the attributes and hours columns into new columns
expanded = expand_nested(intermediate, "attributes")
expanded = expand_nested(expanded, "hours")

Example of value in attributes column: {'OutdoorSeating': 'True', 'RestaurantsPriceRange2': '2', 'BusinessAcceptsCreditCards': 'True', 'DogsAllowed': 'True', 'Ambience': "{'touristy': False, 'hipster': False, 'romantic': False, 'divey': False, 'intimate': False, 'trendy': False, 'upscale': False, 'classy': True, 'casual': False}", 'HappyHour': 'True', 'GoodForMeal': "{'dessert': False, 'latenight': False, 'lunch': False, 'dinner': False, 'brunch': False, 'breakfast': False}", 'RestaurantsDelivery': 'True', 'HasTV': 'True', 'BusinessParking': "{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}", 'RestaurantsTakeOut': 'True', 'GoodForKids': 'True'}
Example of value in hours column: {'Monday': '0:0-0:0', 'Wednesday': '16:0-22:0', 'Thursday': '16:0-22:0', 'Friday': '16:0-19:0', 'Saturday': '11:0-23:0', 'Sunday': '11:0-20:0'}


- Category labels are stored as a string list in column values.
- Hence, this section extract these labels and encode them into dummy variables (new columns)

In [7]:
# each value in the categories column is a list of labels that the restaurant has
print(f"Example of value in categories column: {intermediate["categories"][10]}")

# similarily, we encode each label in the categories column as a dummy variable
# we only keep labels that appear in > 1% restaurants (166 restaurants), to keep feature number accountable
expanded = expand_cats(expanded, "categories", 0.01)

# drop the resturant label as every observations in the dataset is a resturant
expanded = expanded.drop(columns=["Restaurants"])

Example of value in categories column: Eatertainment, Arts & Entertainment, Brewpubs, American (Traditional), Bakeries, Breweries, Food, Restaurants


- get column names and datatypes of the transformed dataset

In [9]:
# we now have 173 columns: 3 are floats (latitude, longitude, stars), 77 are ints (include dummies), 94 are strings or Boolen
print(col_dtype(expanded))

index                              |int64
business_id                        |object
name                               |object
address                            |object
city                               |object
state                              |object
postal_code                        |object
latitude                           |float64
longitude                          |float64
stars                              |float64
review_count                       |int64
is_open                            |int64
attributes                         |object
categories                         |object
hours                              |object
RestaurantsDelivery                |object
OutdoorSeating                     |object
BusinessAcceptsCreditCards         |object
BusinessParking_garage             |object
BusinessParking_street             |object
BusinessParking_validated          |object
BusinessParking_lot                |object
BusinessParking_valet              |object
BikeParking